# 📘 Módulo 04 - Notebook 01: Valores Faltantes y Duplicados

## 🧹 Detección y Manejo de Datos Incompletos

**Libro:** Saliendo de lo Pandito v4  
**Módulo:** 04 - Limpieza y Preparación de Datos  
**Duración estimada:** 70 minutos  
**Dificultad:** 🟡 Intermedio  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos

✅ **Comprender** qué son los valores nulos y por qué ocurren  
✅ **Detectar** valores faltantes con múltiples métodos  
✅ **Analizar** el impacto de los nulos en los datos  
✅ **Aplicar** estrategias de manejo (eliminar vs rellenar)  
✅ **Dominar** métodos de imputación (constante, estadística, ffill, bfill)  
✅ **Identificar** y eliminar registros duplicados  
✅ **Resolver** casos contables reales con datos sucios

---

## 📚 Contenido

1. Introducción a Valores Nulos
2. Detección de Valores Faltantes
3. Análisis del Patrón de Nulos
4. Estrategias de Manejo de Nulos
5. Métodos de Imputación
6. Introducción a Duplicados
7. Detección y Eliminación de Duplicados
8. Caso Contable Integrador
9. Conclusiones y Mejores Prácticas

---

## 💡 Por Qué Importa

**"Los datos reales siempre están sucios."**

En datos empresariales:
* 📊 **Ventas:** Clientes sin email, direcciones incompletas
* 💰 **Finanzas:** Facturas sin fecha de pago, costos faltantes
* 📦 **Inventario:** SKUs sin precio, stock no registrado
* 👥 **RRHH:** Empleados sin departamento, salarios sin fecha

**El 80% del trabajo de análisis es limpiar datos. Dominémoslo.**

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🧹 VALORES FALTANTES Y DUPLICADOS")
print("="*70)
print("• Pandas: Más del 70% de datasets reales tienen valores nulos")
print("• Duplicados: Errores comunes en data entry y ETL")
print("• Objetivo: Limpiar datos antes de cualquier análisis")
print("\n📖 Métodos clave:")
print("  - .isnull() / .isna()    : Detectar nulos")
print("  - .notnull() / .notna()  : Detectar no-nulos")
print("  - .fillna()              : Rellenar nulos")
print("  - .dropna()              : Eliminar nulos")
print("  - .duplicated()          : Detectar duplicados")
print("  - .drop_duplicates()     : Eliminar duplicados")
print("="*70)
print("✅ Librerías cargadas")


## 🔍 ¿Qué son los Valores Nulos?

### 📚 Definición

Un **valor nulo (missing value)** es la **ausencia de dato** en una celda.

**Representaciones en Pandas:**
* `NaN` (Not a Number) - numpy.nan
* `None` - Python None
* `pd.NA` - Pandas NA (nuevo desde v1.0)

---

### 🐛 Causas Comunes

| Causa | Ejemplo Empresarial |
|-------|---------------------|
| **Data entry incompleto** | Cliente registrado sin email |
| **Fallo en integración** | API no devuelve campo opcional |
| **Dato no aplica** | Fecha de pago (factura pendiente) |
| **Sensores fallidos** | Temperatura no registrada |
| **Join con tabla externa** | LEFT JOIN sin match |
| **Datos eliminados** | Producto descontinuado |

---

### ⚠️ Impacto de los Nulos

👎 **Problemas:**
* Cálculos incorrectos (sum, mean ignoran nulos)
* Modelos ML fallan o sesgan
* Joins pierden registros
* Visualizaciones incompletas

👍 **Buena práctica:**
* **SIEMPRE** detectar y analizar nulos primero
* **NUNCA** asumir que no hay nulos
* **DOCUMENTAR** decisión de manejo

---

### 📊 Tipos de Datos Faltantes

**1. MCAR (Missing Completely At Random)**
* Aleatoriedad pura
* Ejemplo: Sensor falla aleatoriamente
* ✅ Safe imputar con promedio

**2. MAR (Missing At Random)**
* Relacionado con otras variables
* Ejemplo: Mujeres no declaran edad
* ⚠️ Imputar con cuidado

**3. MNAR (Missing Not At Random)**
* Relacionado con el valor faltante
* Ejemplo: Ingresos altos no declarados
* 🚫 Imputación simple sesga resultados

In [0]:
import pandas as pd
import numpy as np

print("🔍 DETECCIÓN DE VALORES NULOS")
print("="*70)

# DataFrame con datos contables y valores nulos
df = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=10),
    'Cliente': ['Acme Corp', 'TechStart', None, 'FinPlus', 'LogiExp', 
                'Acme Corp', None, 'DataCo', 'FinPlus', 'TechStart'],
    'Monto': [15000, np.nan, 8900, 12000, np.nan, 
              22000, 9500, np.nan, 14000, 18000],
    'Descuento': [0.10, 0.05, None, 0.0, 0.15,
                  0.10, np.nan, 0.05, 0.0, 0.10],
    'Estado': ['Pagado', 'Pendiente', 'Pagado', None, 'Pagado',
               'Pagado', 'Pendiente', 'Pagado', None, 'Pendiente']
})

print("\n📊 DataFrame original (10 transacciones):")
print(df)
print(f"\nShape: {df.shape}")

print("\n" + "-"*70)
print("\n1️⃣  DETECTAR NULOS POR COLUMNA (.isnull().sum())")
nulos_por_columna = df.isnull().sum()
print(nulos_por_columna)
print(f"\n📊 Interpretación:")
for col, count in nulos_por_columna.items():
    if count > 0:
        pct = (count / len(df)) * 100
        print(f"  • {col}: {count} nulos ({pct:.1f}%)")

print("\n" + "-"*70)
print("\n2️⃣  DETECTAR FILAS CON AL MENOS 1 NULO")
filas_con_nulos = df.isnull().any(axis=1)
print(f"\nFilas con nulos: {filas_con_nulos.sum()} de {len(df)}")
print("\nFilas afectadas:")
print(df[filas_con_nulos][['Cliente', 'Monto', 'Estado']])

print("\n" + "-"*70)
print("\n3️⃣  TOTAL DE VALORES NULOS EN TODO EL DATAFRAME")
total_nulos = df.isnull().sum().sum()
total_celdas = df.shape[0] * df.shape[1]
pct_nulos = (total_nulos / total_celdas) * 100
print(f"\nTotal de nulos: {total_nulos} de {total_celdas} celdas ({pct_nulos:.1f}%)")

print("\n" + "-"*70)
print("\n4️⃣  FILAS COMPLETAMENTE COMPLETAS (.notnull().all(axis=1))")
filas_completas = df.notnull().all(axis=1)
print(f"\nFilas sin nulos: {filas_completas.sum()} de {len(df)}")
print("\nFilas perfectas:")
print(df[filas_completas])

print("\n" + "="*70)
print("✅ Detección completada")

## ⚙️ Estrategias de Manejo de Nulos

### 🔀 Dos Enfoques Principales

#### 1️⃣  ELIMINAR (Deletion)

**Método:** `.dropna()`

👍 **Pros:**
* Simple y rápido
* No introduce sesgo si MCAR
* Datos reales post-eliminación

👎 **Contras:**
* Pérdida de información
* Reduce tamaño del dataset
* Sesgo si MAR o MNAR

**✅ Cuándo usar:**
* <5% de datos faltantes
* Dataset muy grande
* Nulos aleatorios (MCAR)
* Columnas irrelevantes

---

#### 2️⃣  RELLENAR (Imputation)

**Método:** `.fillna()`

👍 **Pros:**
* Mantiene tamaño del dataset
* Preserva relaciones
* Útil para ML

👎 **Contras:**
* Puede introducir sesgo
* Valores no reales
* Reduce varianza

**✅ Cuándo usar:**
* >10% de datos faltantes
* Dataset pequeño
* Nulos sistemáticos
* ML requiere datos completos

---

### 📊 Regla de Oro

```
< 5%    nulos  →  Eliminar (.dropna())
5-20%   nulos  →  Rellenar con criterio
> 20%   nulos  →  Investigar causa, considerar quitar columna
```

---

### 🧠 Métodos de Imputación

| Método | Código | Uso |
|--------|--------|-----|
| **Constante** | `fillna(0)` | Defaults conocidos |
| **Promedio** | `fillna(mean())` | Datos numéricos simétricos |
| **Mediana** | `fillna(median())` | Datos con outliers |
| **Moda** | `fillna(mode()[0])` | Categóricos |
| **Forward fill** | `fillna(method='ffill')` | Series temporales |
| **Backward fill** | `fillna(method='bfill')` | Series temporales |
| **Interpolación** | `interpolate()` | Series temporales suaves |
| **ML** | KNN, MICE | Relaciones complejas |

In [0]:
import pandas as pd
import numpy as np

print("🛠️ ELIMINACIÓN DE VALORES NULOS")
print("="*70)

# DataFrame de ejemplo
df = pd.DataFrame({
    'Cliente': ['Acme', 'TechStart', None, 'FinPlus', 'LogiExp'],
    'Monto': [15000, np.nan, 8900, 12000, np.nan],
    'Descuento': [0.10, 0.05, None, 0.0, 0.15],
    'Estado': ['Pagado', 'Pendiente', 'Pagado', None, 'Pagado']
})

print("\n📊 DataFrame original:")
print(df)
print(f"\nShape: {df.shape}")
print(f"Nulos por columna:\n{df.isnull().sum()}")

print("\n" + "-"*70)
print("\n1️⃣  ELIMINAR FILAS CON CUALQUIER NULO (.dropna())")
df_drop_any = df.dropna()
print(f"\nAntes: {df.shape[0]} filas")
print(f"Después: {df_drop_any.shape[0]} filas")
print(f"Eliminadas: {df.shape[0] - df_drop_any.shape[0]} filas")
print("\nResultado:")
print(df_drop_any)

print("\n" + "-"*70)
print("\n2️⃣  ELIMINAR FILAS DONDE TODAS SON NULAS (.dropna(how='all'))")
df_test = df.copy()
df_test.loc[5] = [None, None, None, None]  # Fila completamente nula
print("\nDataFrame con fila completamente nula:")
print(df_test)
df_drop_all = df_test.dropna(how='all')
print(f"\nAntes: {df_test.shape[0]} filas")
print(f"Después: {df_drop_all.shape[0]} filas")

print("\n" + "-"*70)
print("\n3️⃣  ELIMINAR FILAS CON NULOS EN COLUMNAS ESPECÍFICAS")
print("\nEliminar donde 'Monto' es nulo:")
df_drop_monto = df.dropna(subset=['Monto'])
print(df_drop_monto)
print(f"\nAntes: {df.shape[0]} filas → Después: {df_drop_monto.shape[0]} filas")

print("\n" + "-"*70)
print("\n4️⃣  ELIMINAR COLUMNAS CON NULOS (.dropna(axis=1))")
print("\nEliminar columnas con al menos 1 nulo:")
df_drop_cols = df.dropna(axis=1)
print(df_drop_cols)
print(f"\nColumnas antes: {df.shape[1]} → Después: {df_drop_cols.shape[1]}")

print("\n" + "-"*70)
print("\n5️⃣  UMBRAL MÍNIMO DE VALORES NO-NULOS (.dropna(thresh=))")
print("\nMantener filas con al menos 3 valores no-nulos:")
df_thresh = df.dropna(thresh=3)
print(df_thresh)
print(f"\nAntes: {df.shape[0]} filas → Después: {df_thresh.shape[0]} filas")

print("\n" + "="*70)
print("✅ Eliminación dominada")

In [0]:
import pandas as pd
import numpy as np

print("📊 MÉTODOS DE IMPUTACIÓN")
print("="*70)

# DataFrame con nulos
df = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=10),
    'Producto': ['Laptop'] * 10,
    'Precio': [899, 920, np.nan, 910, np.nan, 905, 915, np.nan, 900, 925],
    'Stock': [15, np.nan, 12, 10, np.nan, 8, 11, 9, np.nan, 14],
    'Categoria': ['A', 'B', None, 'A', 'B', None, 'A', 'B', 'A', None]
})

print("\n📊 DataFrame original con nulos:")
print(df)
print(f"\nNulos por columna:\n{df.isnull().sum()}")

print("\n" + "="*70)
print("\n1️⃣  IMPUTACIÓN CON CONSTANTE")
df_const = df.copy()
df_const['Precio'] = df_const['Precio'].fillna(0)
df_const['Stock'] = df_const['Stock'].fillna(0)
df_const['Categoria'] = df_const['Categoria'].fillna('DESCONOCIDO')
print("\nRellenar con 0 (numérico) y 'DESCONOCIDO' (categórico):")
print(df_const[['Precio', 'Stock', 'Categoria']])

print("\n" + "="*70)
print("\n2️⃣  IMPUTACIÓN CON PROMEDIO")
df_mean = df.copy()
precio_promedio = df['Precio'].mean()
df_mean['Precio'] = df_mean['Precio'].fillna(precio_promedio)
print(f"\nPromedio de Precio: ${precio_promedio:.2f}")
print("\nPrecios después de imputar con promedio:")
print(df_mean['Precio'])

print("\n" + "="*70)
print("\n3️⃣  IMPUTACIÓN CON MEDIANA (mejor para outliers)")
df_median = df.copy()
stock_mediana = df['Stock'].median()
df_median['Stock'] = df_median['Stock'].fillna(stock_mediana)
print(f"\nMediana de Stock: {stock_mediana}")
print("\nStock después de imputar con mediana:")
print(df_median['Stock'])

print("\n" + "="*70)
print("\n4️⃣  IMPUTACIÓN CON MODA (categóricos)")
df_mode = df.copy()
categoria_moda = df['Categoria'].mode()[0]
df_mode['Categoria'] = df_mode['Categoria'].fillna(categoria_moda)
print(f"\nModa de Categoria: '{categoria_moda}'")
print("\nCategoría después de imputar con moda:")
print(df_mode['Categoria'])

print("\n" + "="*70)
print("\n5️⃣  FORWARD FILL (ffill) - Series temporales")
df_ffill = df.copy()
df_ffill['Precio'] = df_ffill['Precio'].fillna(method='ffill')
print("\nPropagar último valor válido hacia adelante:")
print(df_ffill[['Fecha', 'Precio']])

print("\n" + "="*70)
print("\n6️⃣  BACKWARD FILL (bfill) - Series temporales")
df_bfill = df.copy()
df_bfill['Stock'] = df_bfill['Stock'].fillna(method='bfill')
print("\nPropagar siguiente valor válido hacia atrás:")
print(df_bfill[['Fecha', 'Stock']])

print("\n" + "="*70)
print("\n7️⃣  INTERPOLACIÓN LINEAL (series temporales suaves)")
df_interp = df.copy()
df_interp['Precio'] = df_interp['Precio'].interpolate(method='linear')
print("\nInterpolación lineal entre valores conocidos:")
print(df_interp[['Fecha', 'Precio']])

print("\n" + "="*70)
print("✅ Métodos de imputación dominados")

## 🔄 Registros Duplicados

### 📚 Definición

Un **registro duplicado** es una fila que **aparece más de una vez** en el DataFrame.

**Tipos de duplicados:**

1. **Duplicados exactos** - Todas las columnas idénticas
2. **Duplicados parciales** - Solo algunas columnas idénticas (ej: mismo ID)
3. **Duplicados fuzzy** - Similares pero no idénticos ("Acme Corp" vs "ACME CORP")

---

### 🐛 Causas Comunes

| Causa | Ejemplo Empresarial |
|-------|---------------------|
| **Data entry doble** | Usuario presiona "Guardar" 2 veces |
| **ETL mal configurado** | Script corre 2 veces sin dedup |
| **Merge sin dedup** | UNION ALL en vez de UNION |
| **Importación repetida** | Archivo procesado múltiples veces |
| **Claves no únicas** | Tabla sin PRIMARY KEY |

---

### ⚠️ Impacto de Duplicados

👎 **Problemas:**
* Cálculos inflados (sum, count incorrectos)
* Clientes facturados 2 veces
* Inventario sobreestimado
* Reportes engañosos
* Violaciones de integridad

---

### 📊 Métodos de Detección

```python
# Detectar duplicados completos
df.duplicated()  # Series booleana

# Contar duplicados
df.duplicated().sum()

# Ver filas duplicadas
df[df.duplicated(keep=False)]

# Duplicados por columnas específicas
df.duplicated(subset=['ID_Cliente'])
```

---

### 🛠️ Métodos de Eliminación

```python
# Eliminar duplicados (mantener primero)
df.drop_duplicates()

# Mantener último
df.drop_duplicates(keep='last')

# Eliminar todos (no mantener ninguno)
df.drop_duplicates(keep=False)

# Por columnas específicas
df.drop_duplicates(subset=['ID_Cliente'])
```

---

### 🎯 Parámetro `keep`

| Valor | Comportamiento |
|-------|----------------|
| `'first'` (default) | Mantiene primera ocurrencia |
| `'last'` | Mantiene última ocurrencia |
| `False` | Elimina TODAS las ocurrencias |

In [0]:
import pandas as pd
import numpy as np

print("🔄 DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS")
print("="*70)

# DataFrame con duplicados
df = pd.DataFrame({
    'ID_Cliente': [101, 102, 103, 103, 104, 105, 105, 105],
    'Razon_Social': ['Acme Corp', 'TechStart', 'FinPlus', 'FinPlus', 
                     'LogiExp', 'DataCo', 'DataCo', 'DataCo'],
    'Saldo': [45000, 28000, 12000, 12000, 33000, 19000, 19000, 19000],
    'Fecha_Registro': ['2024-01-15', '2024-02-01', '2024-01-20', '2024-01-20',
                       '2024-03-10', '2024-02-15', '2024-02-15', '2024-02-15']
})

print("\n📊 DataFrame original con duplicados:")
print(df)
print(f"\nShape: {df.shape}")

print("\n" + "-"*70)
print("\n1️⃣  DETECTAR DUPLICADOS COMPLETOS (.duplicated())")
duplicados = df.duplicated()
print(f"\nFilas duplicadas: {duplicados.sum()}")
print("\nMáscara booleana:")
print(duplicados)
print("\n👉 True = fila duplicada (ya apareció antes)")

print("\n" + "-"*70)
print("\n2️⃣  VER TODAS LAS FILAS DUPLICADAS (keep=False)")
print("\nIncluye TODAS las ocurrencias (primera y subsecuentes):")
duplicados_todos = df.duplicated(keep=False)
print(df[duplicados_todos])
print(f"\nTotal de filas en duplicación: {duplicados_todos.sum()}")

print("\n" + "-"*70)
print("\n3️⃣  DUPLICADOS POR COLUMNAS ESPECÍFICAS")
print("\nDuplicados solo por 'ID_Cliente':")
duplicados_id = df.duplicated(subset=['ID_Cliente'], keep=False)
print(df[duplicados_id][['ID_Cliente', 'Razon_Social']])

print("\n" + "-"*70)
print("\n4️⃣  ELIMINAR DUPLICADOS (mantener primero - keep='first')")
df_dedup_first = df.drop_duplicates()
print(f"\nAntes: {df.shape[0]} filas")
print(f"Después: {df_dedup_first.shape[0]} filas")
print(f"Eliminadas: {df.shape[0] - df_dedup_first.shape[0]} filas")
print("\nResultado:")
print(df_dedup_first)

print("\n" + "-"*70)
print("\n5️⃣  ELIMINAR DUPLICADOS (mantener último - keep='last')")
df_dedup_last = df.drop_duplicates(keep='last')
print("\nMantiene la Última ocurrencia:")
print(df_dedup_last)

print("\n" + "-"*70)
print("\n6️⃣  ELIMINAR TODOS LOS DUPLICADOS (keep=False)")
df_dedup_none = df.drop_duplicates(keep=False)
print("\nElimina TODAS las ocurrencias (incluso la primera):")
print(df_dedup_none)
print(f"\nSolo quedan {df_dedup_none.shape[0]} registros únicos")

print("\n" + "-"*70)
print("\n7️⃣  DUPLICADOS POR COLUMNA ESPECÍFICA")
print("\nEliminar duplicados solo por 'ID_Cliente':")
df_dedup_id = df.drop_duplicates(subset=['ID_Cliente'], keep='first')
print(df_dedup_id)
print(f"\n👉 Cada ID_Cliente aparece solo 1 vez")

print("\n" + "="*70)
print("✅ Duplicados dominados")

In [0]:
import pandas as pd
import numpy as np

print("💼 CASO INTEGRADOR: LIMPIEZA COMPLETA DE CUENTAS POR COBRAR")
print("="*70)

# Dataset realista con múltiples problemas
np.random.seed(42)
df_raw = pd.DataFrame({
    'ID_Factura': ['F001', 'F002', 'F003', 'F003', 'F004', 'F005', 'F006', 'F006', 'F007', 'F008',
                   'F009', 'F010', 'F011', 'F012', 'F013', 'F014', 'F015'],
    'Cliente': ['Acme Corp', ' ACME CORP ', 'TechStart', 'TechStart', None, 'FinPlus', 
                'LogiExp', 'LogiExp', 'DataCo', None, 'Acme Corp', 'TechStart', 
                'FinPlus', 'NewCo', 'OldCo', None, 'FastCo'],
    'Monto': [45000, 45000, np.nan, 28000, 12000, np.nan, 33000, 33000, 19000, 21000,
              np.nan, 15000, 18000, np.nan, 25000, 31000, np.nan],
    'Fecha_Vencimiento': pd.to_datetime(['2024-01-15', '2024-01-15', '2024-02-01', '2024-02-01',
                                         '2024-01-20', None, '2024-03-10', '2024-03-10',
                                         '2024-02-15', '2024-04-01', '2024-01-25', '2024-02-20',
                                         None, '2024-03-15', '2024-01-30', '2024-02-28', '2024-03-20']),
    'Estado': ['Pendiente', 'Pendiente', 'Pagado', 'Pagado', None, 'Pendiente', 
               'Vencido', 'Vencido', 'Pagado', 'Pendiente', 'Vencido', 'Pagado',
               'Pendiente', None, 'Vencido', 'Pendiente', 'Pagado']
})

print("\n📄 DATOS ORIGINALES (17 facturas con problemas):")
print(df_raw)
print(f"\nShape inicial: {df_raw.shape}")

print("\n" + "="*70)
print("\n🔍 DIAGNÓSTICO INICIAL")
print("-"*70)
print("\n1. Valores nulos por columna:")
print(df_raw.isnull().sum())

print("\n2. Duplicados completos:")
print(f"  • Filas duplicadas: {df_raw.duplicated().sum()}")
print(f"  • Total en duplicación: {df_raw.duplicated(keep=False).sum()}")

print("\n3. Duplicados por ID_Factura:")
duplicados_id = df_raw.duplicated(subset=['ID_Factura'], keep=False)
print(f"  • IDs duplicados: {duplicados_id.sum()}")
if duplicados_id.sum() > 0:
    print("\n  Facturas duplicadas:")
    print(df_raw[duplicados_id][['ID_Factura', 'Cliente', 'Monto']].to_string())

print("\n" + "="*70)
print("\n🧹 PROCESO DE LIMPIEZA")
print("-"*70)

df_limpio = df_raw.copy()

print("\nPaso 1: Eliminar duplicados exactos (mantener primero)")
antes = df_limpio.shape[0]
df_limpio = df_limpio.drop_duplicates()
print(f"  • Eliminadas: {antes - df_limpio.shape[0]} filas")
print(f"  • Filas actuales: {df_limpio.shape[0]}")

print("\nPaso 2: Imputar Monto con mediana (mejor que promedio para outliers)")
monto_mediana = df_limpio['Monto'].median()
print(f"  • Mediana de Monto: ${monto_mediana:,.0f}")
nulos_monto = df_limpio['Monto'].isnull().sum()
df_limpio['Monto'] = df_limpio['Monto'].fillna(monto_mediana)
print(f"  • Nulos rellenados: {nulos_monto}")

print("\nPaso 3: Eliminar filas donde Cliente es nulo (crítico)")
antes = df_limpio.shape[0]
df_limpio = df_limpio.dropna(subset=['Cliente'])
print(f"  • Eliminadas: {antes - df_limpio.shape[0]} filas")

print("\nPaso 4: Normalizar nombres de Cliente (espacios, mayúsculas)")
df_limpio['Cliente'] = df_limpio['Cliente'].str.strip().str.upper()
print("  • Clientes normalizados")

print("\nPaso 5: Imputar Estado con moda (categórico)")
estado_moda = df_limpio['Estado'].mode()[0]
print(f"  • Moda de Estado: '{estado_moda}'")
nulos_estado = df_limpio['Estado'].isnull().sum()
df_limpio['Estado'] = df_limpio['Estado'].fillna(estado_moda)
print(f"  • Nulos rellenados: {nulos_estado}")

print("\nPaso 6: Eliminar filas donde Fecha_Vencimiento es nula")
antes = df_limpio.shape[0]
df_limpio = df_limpio.dropna(subset=['Fecha_Vencimiento'])
print(f"  • Eliminadas: {antes - df_limpio.shape[0]} filas")

print("\n" + "="*70)
print("\n✅ DATOS LIMPIOS")
print("-"*70)
print(df_limpio)
print(f"\nShape final: {df_limpio.shape}")
print(f"\n📊 Resumen de limpieza:")
print(f"  • Filas originales: {df_raw.shape[0]}")
print(f"  • Filas finales: {df_limpio.shape[0]}")
print(f"  • Filas eliminadas: {df_raw.shape[0] - df_limpio.shape[0]} ({((df_raw.shape[0] - df_limpio.shape[0])/df_raw.shape[0]*100):.1f}%)")
print(f"  • Nulos restantes: {df_limpio.isnull().sum().sum()}")

print("\n📊 Análisis post-limpieza:")
print(f"  • Total por cobrar: ${df_limpio['Monto'].sum():,.0f}")
print(f"  • Monto promedio: ${df_limpio['Monto'].mean():,.0f}")
print(f"  • Clientes únicos: {df_limpio['Cliente'].nunique()}")
print(f"  • Estados: {df_limpio['Estado'].value_counts().to_dict()}")

print("\n" + "="*70)
print("✅ Caso integrador completado")

## 🎓 Conclusiones del Notebook 04_01

### ✅ Lo Que Aprendiste

1. **Valores Nulos:**
   - ¿Qué son y por qué ocurren? (MCAR, MAR, MNAR)
   - Detección: `.isnull()`, `.notnull()`, `.isnull().sum()`
   - Eliminación: `.dropna()` con parámetros (how, subset, thresh, axis)
   - Imputación: constante, promedio, mediana, moda, ffill, bfill, interpolate

2. **Duplicados:**
   - Tipos: exactos, parciales, fuzzy
   - Detección: `.duplicated()` con keep (first, last, False)
   - Eliminación: `.drop_duplicates()` con subset y keep
   - Impacto en cálculos y reportes

3. **Caso Integrador:**
   - Pipeline completo de limpieza
   - Combinación de técnicas
   - Toma de decisiones basada en contexto empresarial

---

### 🎯 Conceptos Clave

🧹 **Regla de Oro:**
```
< 5%    nulos  →  Eliminar (.dropna())
5-20%   nulos  →  Rellenar con criterio
> 20%   nulos  →  Investigar causa
```

📊 **Métodos de Imputación:**
* **Numéricos:** promedio (simétrico), mediana (outliers)
* **Categóricos:** moda
* **Series temporales:** ffill, bfill, interpolate
* **Constante:** 0, -1, "DESCONOCIDO"

🔄 **Duplicados:**
* `keep='first'` : Mantiene primera ocurrencia (default)
* `keep='last'`  : Mantiene última ocurrencia
* `keep=False`   : Elimina TODAS las ocurrencias

---

### 🚀 Próximo Notebook

**04_02 - Imputación Estadística Avanzada**
* Imputación multivariada (KNN, MICE)
* Imputación por grupos (GroupBy + fillna)
* Evaluación de calidad de imputación
* Casos con relaciones complejas

---

<div style="background: linear-gradient(90deg, #7c3aed 0%, #a78bfa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🧹 ¡Limpieza Básica Dominada!</h3>
  <p><i>"Los datos limpios son la base de todo análisis confiable. Ahora pasemos a técnicas avanzadas."</i></p>
</div>